In [ ]:
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")


# Phase 1: Data Inspection


In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os

# Download the Hindi NER dataset ZIP file from Hugging Face
z = hf_hub_download(
    "ai4bharat/naamapadam",  # Hugging Face dataset repository
    "data/hi_IndicNER_v1.0.zip",  # Path of the ZIP file inside the repository
    repo_type="dataset",  # Specify that the repository contains a dataset
)

# Define the directory where the extracted files will be stored
d = z + ".x"

# Extract the ZIP file only if the extraction directory doesn't already exist
if not os.path.isdir(d):
    with zipfile.ZipFile(z) as f:
        f.extractall(d)  # Extract all files into the specified directory

# Display the names of files and folders inside the extraction directory
print(os.listdir(d))


['hi_test.json', 'hi_train.json', 'hi_val.json']


In [ ]:
import json

# Construct the path to the Hindi training dataset JSON file
train = os.path.join(d, "hi_train.json")

# Open the JSON file using UTF-8 encoding
with open(train, encoding="utf-8") as f:
    # Read only the first line of the file
    first = f.readline()

# Print the first line to inspect the dataset format
print(first)


{"words": ["सेक्टर", "55/56", "के", "एसएचओ", "अरविंद", "कुमार", "ने", "बताया", "कि", "इस", "मामले", "में", "आईपीसी", "की", "धारा", "376", "-", "डी", "(", "गैंगरेप", ")", "के", "तहत", "मामला", "दर्ज", "कर", "लिया", "गया", "है", "।"], "ner": ["O", "O", "O", "O", "B-PER", "I-PER", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O"]}



In [ ]:
# Define a function to load the dataset from a JSONL file
def load(path):
    rows = []

    # Open the file using UTF-8 encoding
    with open(path, encoding="utf-8") as f:

        # Read the file one line at a time
        for line in f:
            line = line.strip()  # Remove leading/trailing whitespace

            # Skip empty lines
            if line:
                r = json.loads(line)  # Parse the JSON string into a Python dictionary

                # Store the words (tokens) and their corresponding NER labels
                rows.append((r["words"], r["ner"]))

    # Return all loaded sentences and their NER labels
    return rows


# Load Hindi training, validation, and testing datasets
hi = {s: load(os.path.join(d, f"hi_{s}.json")) for s in ["train", "val", "test"]}

# Print the number of sentences in each dataset split
for s, rows in hi.items():
    print(s, len(rows))


train 985787
val 13460
test 867


In [ ]:
from collections import Counter

# Analyze NER labels and data quality for each dataset split
for s, rows in hi.items():
    tags = Counter()  # Count occurrences of each NER tag
    mismatch = 0  # Count sentences where words and tags have different lengths
    empty = 0  # Count empty sentences

    for w, n in rows:
        tags.update(n)  # Update tag frequencies

        # Check whether the number of words matches the number of NER tags
        if len(w) != len(n):
            mismatch += 1

        # Check whether the sentence contains no words
        if len(w) == 0:
            empty += 1

    # Print dataset statistics
    print(f"\n{'=' * 50}")
    print(f"Dataset Split: {s.upper()}")
    print(f"{'=' * 50}")
    print(f"Total sentences       : {len(rows):,}")
    print(f"Total NER tags        : {sum(tags.values()):,}")
    print(f"Length mismatches     : {mismatch:,}")
    print(f"Empty sentences       : {empty:,}")

    print("\nNER Tag Distribution:")
    for tag, count in sorted(tags.items()):
        print(f"  {tag:<15}: {count:>10,}")

    print(f"{'=' * 50}")



Dataset Split: TRAIN
Total sentences       : 985,787
Total NER tags        : 22,029,408
Length mismatches     : 0
Empty sentences       : 0

NER Tag Distribution:
  B-LOC          :    731,183
  B-ORG          :    686,388
  B-PER          :    767,003
  I-LOC          :    207,240
  I-ORG          :    825,713
  I-PER          :    712,403
  O              : 18,099,478

Dataset Split: VAL
Total sentences       : 13,460
Total NER tags        : 304,282
Length mismatches     : 0
Empty sentences       : 0

NER Tag Distribution:
  B-LOC          :     10,209
  B-ORG          :      9,735
  B-PER          :     10,549
  I-LOC          :      2,779
  I-ORG          :     11,506
  I-PER          :      9,870
  O              :    249,634

Dataset Split: TEST
Total sentences       : 867
Total NER tags        : 19,893
Length mismatches     : 0
Empty sentences       : 3

NER Tag Distribution:
  B-LOC          :        613
  B-ORG          :        521
  B-PER          :        788
  I-LOC      

In [ ]:
import numpy as np

# Calculate the number of tokens in each training sentence
lens = np.array([len(w) for w, _ in hi["train"]])

# Calculate and display sentence length statistics
print("\nTraining Dataset Sentence Length Statistics")
print("-" * 45)
print(f"Minimum length  : {lens.min()}")
print(f"Median length   : {int(np.median(lens))}")
print(f"95th percentile : {int(np.percentile(lens, 95))}")
print(f"99th percentile : {int(np.percentile(lens, 99))}")
print(f"Maximum length  : {lens.max()}")
print("-" * 45)



Training Dataset Sentence Length Statistics
---------------------------------------------
Minimum length  : 1
Median length   : 20
95th percentile : 48
99th percentile : 67
Maximum length  : 100
---------------------------------------------


In [ ]:
# Check whether the NER labels follow the IOB2 tagging scheme
def iob2_violations(rows):
    bad = 0  # Count invalid I- tags

    # Iterate through each sentence and its corresponding NER tags
    for _, n in rows:
        prev = "O"  # Initialize previous tag as Outside

        for t in n:
            # An I- tag must follow a B- or I- tag of the same entity type
            if t.startswith("I-"):
                if prev == "O" or prev[2:] != t[2:]:
                    bad += 1

            # Update the previous tag
            prev = t

    # Return the total number of IOB2 violations
    return bad


# Check IOB2 violations in training, validation, and test datasets
print("\nIOB2 Tagging Validation")
print("-" * 45)

for s, rows in hi.items():
    violations = iob2_violations(rows)
    print(f"{s.upper():<10} | Orphan I- tags: {violations:,}")

print("-" * 45)



IOB2 Tagging Validation
---------------------------------------------
TRAIN      | Orphan I- tags: 5,513
VAL        | Orphan I- tags: 74
TEST       | Orphan I- tags: 7
---------------------------------------------


## Findings

Source: `ai4bharat/naamapadam`, Hindi splits, inspected directly from the distributed JSON files.

- Tag set: exactly `O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC` (IOB2). No other labels present.
- `words` and `ner` are always equal length (0 mismatches across all splits).
- Row counts: train 985,787 / val 13,460 / test 867.
- 3 empty rows in test, 0 in train/val.
- Word length: median 20, p95 48, p99 67, max 100.
- Orphan `I-` tags (continuation with no matching start): train 5,513, val 74, test 7 (~0.5%).

### Contract decisions

- Label noise: leave orphan `I-` tags as-is (keeps parity with the published benchmark; seqeval handles them).
- `max_length`: 128 subword tokens.
- Loader drops zero-length sentences on every split.
